# Eagle3 Dataset Preparation for Qwen3-VL

This notebook prepares vision-language datasets for Eagle3 training:
- Downloads ShareGPT4V (English) - 50K samples
- Downloads InternVL/M3IT (Chinese) - 50K samples  
- Filters images by size (<5MB)
- Validates all images
- Saves to Google Drive

**Estimated time:** 3-5 hours (CPU only)
**Required  Drive space:** ~100GB

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone AngelSlim repository (if not already)
import os
if not os.path.exists('/content/AngelSlim'):
    !git clone https://github.com/Tencent/AngelSlim.git /content/AngelSlim
    !cd /content/AngelSlim && git checkout feat/eagle3-qwen3vl-colab-offline

%cd /content/AngelSlim

In [ ]:
# Install dependencies
!pip install -q datasets pillow requests tqdm

In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content/AngelSlim')

## Configuration

In [ ]:
# Dataset configuration
CONFIG = {
    # Google Drive paths
    'drive_root': '/content/drive/MyDrive/Eagle3_Qwen3VL',
    
    # English dataset (ShareGPT4V)
    'english_dataset': {
        'type': 'sharegpt4v',
        'output_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/sharegpt4v_en',
        'num_samples': 50000,
    },
    
    # Chinese dataset (M3IT - more accessible than InternVL)
    'chinese_dataset': {
        'type': 'm3it',
        'output_dir': '/content/drive/MyDrive/Eagle3_Qwen3VL/datasets/m3it_zh',
        'num_samples': 50000,
    },
    
    # Download settings
    'image_max_size_mb': 5.0,
    'num_workers': 4,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Download English Dataset (ShareGPT4V)

In [ ]:
from colab_code.utils import DatasetDownloader

print("=" * 60)
print("Downloading ShareGPT4V (English) dataset")
print("=" * 60)

en_downloader = DatasetDownloader(
    output_dir=CONFIG['english_dataset']['output_dir'],
    image_max_size_mb=CONFIG['image_max_size_mb'],
    num_workers=CONFIG['num_workers'],
)

en_successful, en_failed = en_downloader.download_sharegpt4v(
    num_samples=CONFIG['english_dataset']['num_samples']
)

print(f"\n✅ English dataset: {en_successful} successful, {en_failed} failed")
print(f"Success rate: {en_successful / (en_successful + en_failed) * 100:.1f}%")

In [ ]:
# Validate English dataset
print("\nValidating English dataset...")
en_stats = en_downloader.validate_dataset()

print(f"\nEnglish dataset statistics:")
print(f"  Total samples: {en_stats['total_samples']}")
print(f"  Valid images: {en_stats['valid_images']}")
print(f"  Invalid images: {en_stats['invalid_images']}")
print(f"  Missing images: {en_stats['missing_images']}")

## Download Chinese Dataset (M3IT)

In [ ]:
print("=" * 60)
print("Downloading M3IT (Chinese) dataset")
print("=" * 60)

zh_downloader = DatasetDownloader(
    output_dir=CONFIG['chinese_dataset']['output_dir'],
    image_max_size_mb=CONFIG['image_max_size_mb'],
    num_workers=CONFIG['num_workers'],
)

zh_successful, zh_failed = zh_downloader.download_m3it_chinese(
    num_samples=CONFIG['chinese_dataset']['num_samples']
)

print(f"\n✅ Chinese dataset: {zh_successful} successful, {zh_failed} failed")
print(f"Success rate: {zh_successful / (zh_successful + zh_failed) * 100:.1f}%")

In [ ]:
# Validate Chinese dataset
print("\nValidating Chinese dataset...")
zh_stats = zh_downloader.validate_dataset()

print(f"\nChinese dataset statistics:")
print(f"  Total samples: {zh_stats['total_samples']}")
print(f"  Valid images: {zh_stats['valid_images']}")
print(f"  Invalid images: {zh_stats['invalid_images']}")
print(f"  Missing images: {zh_stats['missing_images']}")

## Summary and Next Steps

In [ ]:
# Calculate total statistics
total_samples = en_stats['valid_images'] + zh_stats['valid_images']
total_size_gb = (en_successful + zh_successful) * 0.001  # Rough estimate

print("=" * 60)
print("DATASET PREPARATION COMPLETE")
print("=" * 60)

print(f"\n📊 Summary:")
print(f"  English samples: {en_stats['valid_images']}")
print(f"  Chinese samples: {zh_stats['valid_images']}")
print(f"  Total samples: {total_samples}")
print(f"  Language distribution: {en_stats['valid_images']/total_samples*100:.1f}% EN / {zh_stats['valid_images']/total_samples*100:.1f}% ZH")
print(f"  Estimated size: ~{total_size_gb:.1f}GB")

print(f"\n📁 Saved to:")
print(f"  English: {CONFIG['english_dataset']['output_dir']}")
print(f"  Chinese: {CONFIG['chinese_dataset']['output_dir']}")

print(f"\n✅ Next steps:")
print(f"  1. Run generate_responses.ipynb to generate VLM responses")
print(f"  2. Run generate_hidden_states.ipynb to extract hidden states")
print(f"  3. Run eagle3_qwen3vl_training_offline.ipynb for training")

## Optional: Visualize Sample

In [ ]:
# Show a random sample from each dataset
import json
import random
from PIL import Image
import matplotlib.pyplot as plt

def show_sample(dataset_dir, title):
    data_file = f"{dataset_dir}/data_raw.jsonl"
    
    # Read random sample
    with open(data_file, 'r') as f:
        lines = f.readlines()
        sample = json.loads(random.choice(lines))
    
    # Load image
    img_path = f"{dataset_dir}/{sample['img_path'].lstrip('./')}"
    img = Image.open(img_path)
    
    # Display
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title(f"{title}\nQuestion: {sample['question']}")
    plt.axis('off')
    plt.show()

# Show English sample
show_sample(CONFIG['english_dataset']['output_dir'], "English Sample (ShareGPT4V)")

# Show Chinese sample
show_sample(CONFIG['chinese_dataset']['output_dir'], "Chinese Sample (M3IT)")